# Offline inference
Internet off. Loads the packaged checkpoint and scores every study in the mounted test metadata.

In [ ]:
from pathlib import Path
import importlib.util
import sys

asset = None
kaggle_input = Path('/kaggle/input')
if kaggle_input.is_dir():
    for child in kaggle_input.iterdir():
        boot = child / 'bootstrap.py'
        if boot.is_file() and (child / 'assets' / 'checkpoint.pt').is_file():
            spec = importlib.util.spec_from_file_location('rsna_asset_bootstrap', boot)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            asset = module.mount_asset(child)
            break
if asset is None:
    raise SystemExit('BLOCKED: versioned code/weights asset is not mounted')
sys.path.insert(0, str(asset['src']))
from rsna_knee.data.metadata import discover_input_root
from rsna_knee.submission.offline import run_offline_inference

data = discover_input_root()
if data is None:
    raise SystemExit('BLOCKED: competition input is not mounted')
result = run_offline_inference(
    checkpoint=Path(asset['checkpoint']),
    series_csv=data / 'test_series.csv',
    dicom_root=data / 'test',
    study_csv=data / 'test.csv',
    dest=Path('/kaggle/working/submission.csv'),
    work=Path('/kaggle/working/infer_cache'),
)
print(result['n_studies'], result['runtime'])
